# RoadWatch – 01 Baseline (YOLOv8n)

Baseline road-damage detector trained on Manasvi's group-aware dataset (`RoadWatch_YOLO_Grouped`).

| ID | Class |
|---|---|
| 0 | D00 – Longitudinal crack |
| 1 | D10 – Transverse crack |
| 2 | D20 – Alligator crack |
| 3 | D40 – Pothole |

**Before you start:** *Runtime → Change runtime type → T4 GPU*, then run the cells from top to bottom.

Training runs are saved to Google Drive (`MyDrive/RoadWatch/runs`), so they survive a Colab disconnect.

## a) Install packages and mount Google Drive

In [ ]:
# Ultralytics provides YOLOv8; gdown downloads files from Google Drive links.
!pip install -q ultralytics gdown

# Mount Drive so training results are stored persistently.
from google.colab import drive
drive.mount('/content/drive')

# Check that a GPU is attached (should list a Tesla T4).
!nvidia-smi -L

## b) Download and unzip the dataset

> **If `gdown` fails** (quota exceeded / permission error): open the shared file in Google Drive,
> choose **"Add shortcut to Drive"**, and run the fallback cell below instead. It copies the zip from
> `/content/drive/MyDrive`.

In [ ]:
# Download the grouped dataset zip from Google Drive.
!gdown 1kIe0dMfRJWRbKxM2ZYB4_9GiU8pCYs7F -O /content/RoadWatch_YOLO_Grouped.zip

# Unzip into /content/data (-q quiet, -o overwrite if re-running).
!unzip -q -o /content/RoadWatch_YOLO_Grouped.zip -d /content/data

In [ ]:
# FALLBACK – only if gdown failed. Adjust the path if the shortcut is in a sub-folder of MyDrive.
# !cp "/content/drive/MyDrive/RoadWatch_YOLO_Grouped.zip" /content/RoadWatch_YOLO_Grouped.zip
# !unzip -q -o /content/RoadWatch_YOLO_Grouped.zip -d /content/data

## c) Find the dataset root and check the splits

The zip may contain an extra top-level folder, so we search for the folder that contains `images/train`.
Expected image counts (from the dataset README): **train ≈ 5001, val ≈ 1429, test ≈ 714**.

In [ ]:
from pathlib import Path

DATA_DIR = Path('/content/data')
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
EXPECTED = {'train': 5001, 'val': 1429, 'test': 714}

# The dataset root is the folder whose images/train sub-folder exists.
candidates = [p.parent.parent for p in DATA_DIR.rglob('images/train') if p.is_dir()]
if not candidates:
    raise FileNotFoundError(f'No images/train folder found under {DATA_DIR}. Did the unzip work?')
DATASET_ROOT = min(candidates, key=lambda p: len(p.parts))  # shallowest match
print('Dataset root:', DATASET_ROOT)

for split in ['train', 'val', 'test']:
    images = [f for f in (DATASET_ROOT / 'images' / split).iterdir() if f.suffix.lower() in IMAGE_EXTS]
    labels = list((DATASET_ROOT / 'labels' / split).glob('*.txt'))
    empty = sum(1 for f in labels if f.stat().st_size == 0)  # negative images have empty label files
    status = 'OK' if len(images) == EXPECTED[split] else f'WARNING: expected about {EXPECTED[split]}'
    print(f'{split:5s}: {len(images):5d} images | {len(labels):5d} labels ({empty} empty = negatives) | {status}')

## d) Write `data.yaml`

In [ ]:
import yaml

DATA_YAML = '/content/data.yaml'
data_config = {
    'path': str(DATASET_ROOT),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 4,
    'names': ['D00', 'D10', 'D20', 'D40'],
}
with open(DATA_YAML, 'w') as f:
    yaml.safe_dump(data_config, f, sort_keys=False)

print(open(DATA_YAML).read())

## e) Train the YOLOv8n baseline

About 50 epochs on a T4. Early stopping (`patience=15`) ends training if validation mAP stops improving.
If a folder `v8n_baseline` already exists, Ultralytics saves to `v8n_baseline2`, `v8n_baseline3`, …

In [ ]:
from ultralytics import YOLO

PROJECT = '/content/drive/MyDrive/RoadWatch/runs'
RUN_NAME = 'v8n_baseline'

model = YOLO('yolov8n.pt')  # COCO-pretrained starting point
results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    seed=42,
    patience=15,
    project=PROJECT,
    name=RUN_NAME,
)
RUN_DIR = Path(results.save_dir)  # actual folder (may be v8n_baseline2 etc.)
print('Run saved to:', RUN_DIR)

## f) Resume after a Colab disconnect

If the runtime disconnects mid-training:

1. Reconnect (T4 GPU) and re-run cells **a**, **b**, **c** and **d**. The resumed run reads the dataset
   from `/content/data.yaml`, so the dataset must be unzipped and the yaml re-created first.
2. Run the cell below. It continues from `last.pt` with the original settings (epochs, seed, …).

Change `RUN_NAME` if your run was saved as `v8n_baseline2`, etc.

In [ ]:
from pathlib import Path
from ultralytics import YOLO

PROJECT = '/content/drive/MyDrive/RoadWatch/runs'
RUN_NAME = 'v8n_baseline'
last_pt = Path(PROJECT) / RUN_NAME / 'weights' / 'last.pt'

if last_pt.exists():
    model = YOLO(str(last_pt))
    results = model.train(resume=True)  # continues the same run in the same folder
    RUN_DIR = Path(results.save_dir)
else:
    print(f'No checkpoint at {last_pt} – nothing to resume.')

## g) Evaluate `best.pt` on the test split

The test split was not used for training or model selection, so these are the numbers to report.

In [ ]:
import json
from pathlib import Path
from ultralytics import YOLO

PROJECT = '/content/drive/MyDrive/RoadWatch/runs'
RUN_NAME = 'v8n_baseline'
RUN_DIR = Path(PROJECT) / RUN_NAME          # change if your run is v8n_baseline2, etc.
best_pt = RUN_DIR / 'weights' / 'best.pt'

metrics = YOLO(str(best_pt)).val(
    data='/content/data.yaml',
    split='test',
    imgsz=640,
    batch=16,
    project=PROJECT,
    name=f'{RUN_NAME}_test',
)

box = metrics.box
print(f'mAP50     : {box.map50:.4f}')
print(f'mAP50-95  : {box.map:.4f}')
print(f'Precision : {box.mp:.4f}')
print(f'Recall    : {box.mr:.4f}')

# Per-class AP50. ap_class_index lists the classes that appear in the test split, in the same order as ap50.
print('\nPer-class AP50:')
per_class = {}
for i, class_id in enumerate(box.ap_class_index):
    name = metrics.names[int(class_id)]
    per_class[name] = round(float(box.ap50[i]), 4)
    print(f'  {name}: {box.ap50[i]:.4f}')

# Keep a copy next to the run on Drive (copy it into model_training/reports/ for the repo).
summary = {
    'weights': str(best_pt),
    'split': 'test',
    'mAP50': round(float(box.map50), 4),
    'mAP50_95': round(float(box.map), 4),
    'precision': round(float(box.mp), 4),
    'recall': round(float(box.mr), 4),
    'per_class_AP50': per_class,
}
out = RUN_DIR / 'test_metrics.json'
out.write_text(json.dumps(summary, indent=2))
print('\nSaved', out)